# Solution: CIF Extraction Pipeline

**V1 (baseline):** single-shot extraction — one LLM call per transcript, full schema in system prompt.  
**V2 (improved):** per-section extraction — parallel focused calls, one per CIF section.  

Dataset: 68 validated synthetic transcripts (12 of 80 excluded due to known transcript defects).

In [1]:
import asyncio
import json
import statistics
from collections import defaultdict
from pathlib import Path

from openai import AsyncOpenAI
from tqdm.notebook import tqdm

from src.eval.models import CIFExtraction
from src.eval.scoring import score_extractions_async
from src.extraction.utils import (
    any_nonnull, cache_load_extractions, cache_load_scores,
    cache_save_extractions, cache_save_scores, count_leaves,
    gt_section_nonempty, holdout_metrics, load_passing_cases,
    make_scoring_input, prf_metrics, section_scores_from,
    strip_front_matter,
)

In [2]:
client = AsyncOpenAI(api_key=Path("openai_creds.txt").read_text().strip())

In [3]:
# ── Model configuration ───────────────────────────────────────────────────
EXTRACTION_MODEL       = "gpt-5.1"
SCORING_MODEL          = "gpt-5.4-mini"
VALIDATION_MODEL       = "gpt-5.4-mini"

EXTRACTION_TEMPERATURE = 0.0
SCORING_TEMPERATURE    = 0.0
VALIDATION_TEMPERATURE = 0.0

_MODEL_SLUG = EXTRACTION_MODEL.replace("/", "-")
print(f"extraction : {EXTRACTION_MODEL}  t={EXTRACTION_TEMPERATURE}")
print(f"scoring    : {SCORING_MODEL}  t={SCORING_TEMPERATURE}")
print(f"validation : {VALIDATION_MODEL}  t={VALIDATION_TEMPERATURE}")

extraction : gpt-5.1  t=0.0
scoring    : gpt-5.4-mini  t=0.0
validation : gpt-5.4-mini  t=0.0


## 1. Load data

In [4]:
cases = load_passing_cases()

by_diff = defaultdict(int)
for c in cases:
    by_diff[c["difficulty"]] += 1

print(f"{len(cases)} passing cases")
for d, n in sorted(by_diff.items()):
    print(f"  {d}: {n}")

68 passing cases
  easy: 19
  hard: 22
  medium: 27


## 2. Preprocessing

In [ ]:
sample = strip_front_matter(cases[0]["transcript"])
print(f"Transcript preprocessing OK — first line: {sample.splitlines()[0][:60]!r}")

Transcript preprocessing OK — first line: 'ADVISOR: [00:00:03] Hi Leila, thanks for making time today. '


## 3. V1 — Single-shot Extraction

One LLM call per transcript. The full `CIFExtraction` JSON Schema is embedded in the system
prompt

In [6]:
from src.eval.utils import load_prompt_yaml

_v1_prompts = load_prompt_yaml(Path("src/prompts/extraction/v1/system.yaml"))
_v1_user   = load_prompt_yaml(Path("src/prompts/extraction/v1/user.yaml"))

_SCHEMA_JSON = json.dumps(CIFExtraction.model_json_schema(), indent=2)

EXTRACTION_SYSTEM_V1 = _v1_prompts["prompt"].format(schema_json=_SCHEMA_JSON)

def make_extraction_user_prompt(transcript: str) -> str:
    return _v1_user["prompt"].format(transcript=strip_front_matter(transcript))

In [7]:
async def extract_cif_v1(
    client: AsyncOpenAI,
    example_id: str,
    transcript: str,
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
    max_retries: int = 3,
) -> dict:
    request = {
        "model": model,
        "input": [
            {"role": "system", "content": EXTRACTION_SYSTEM_V1},
            {"role": "user",   "content": make_extraction_user_prompt(transcript)},
        ],
        "temperature": temperature,
        "text": {"format": {"type": "json_object"}},
        "timeout": 120.0,
    }
    last_error = None
    for attempt in range(max_retries):
        try:
            response = await client.responses.create(**request)
            extracted = CIFExtraction.model_validate_json(response.output_text)
            return {"example_id": example_id, "extracted": extracted, "error": None}
        except Exception as exc:
            last_error = exc
            if attempt < max_retries - 1:
                await asyncio.sleep(2 ** attempt)
    return {"example_id": example_id, "extracted": CIFExtraction(), "error": str(last_error)}


async def extract_cifs_async(
    client: AsyncOpenAI,
    cases: list[dict],
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
    max_concurrency: int = 8,
) -> list[dict]:
    semaphore = asyncio.Semaphore(max_concurrency)

    async def _one(case: dict, pbar: tqdm) -> dict:
        async with semaphore:
            result = await extract_cif_v1(
                client, case["example_id"], case["transcript"], model=model, temperature=temperature
            )
            pbar.update(1)
            if result["error"]:
                pbar.write(f"  ERROR {case['example_id']}: {str(result['error'])[:80]}")
            return result

    with tqdm(total=len(cases), desc="Extracting", unit="case") as pbar:
        return await asyncio.gather(*[_one(c, pbar) for c in cases])

In [8]:
_CACHE = Path(f"data/cache/v1_results_{_MODEL_SLUG}.json")
v1_results = cache_load_extractions(_CACHE)
if v1_results is None:
    v1_results = await extract_cifs_async(
        client, cases, model=EXTRACTION_MODEL, temperature=EXTRACTION_TEMPERATURE
    )
    cache_save_extractions(_CACHE, v1_results)

errors = [r for r in v1_results if r["error"]]
print(f"V1: {len(v1_results) - len(errors)}/{len(v1_results)} ok")
for r in errors:
    print(f"  {r['example_id']}: {r['error']}")

V1: 68/68 ok


In [ ]:
## add validation step after extraction

_val_prompts = {
    "system": load_prompt_yaml(Path("src/prompts/extraction/validate/system.yaml"))["prompt"],
    "user":   load_prompt_yaml(Path("src/prompts/extraction/validate/user.yaml"))["prompt"],
}


async def validate_cif(
    client: AsyncOpenAI,
    example_id: str,
    transcript: str,
    extracted: CIFExtraction,
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
    max_retries: int = 3,
) -> dict:
    """Run a second LLM pass to remove hallucinated fields/items from an extraction."""
    extracted_json = json.dumps(
        extracted.model_dump(mode="json", exclude_none=True), indent=2, ensure_ascii=False
    )
    request = {
        "model": model,
        "input": [
            {"role": "system", "content": _val_prompts["system"]},
            {"role": "user",   "content": _val_prompts["user"].format(
                transcript=strip_front_matter(transcript),
                extracted_json=extracted_json,
            )},
        ],
        "temperature": temperature,
        "text": {"format": {"type": "json_object"}},
        "timeout": 120.0,
    }
    last_error = None
    for attempt in range(max_retries):
        try:
            response = await client.responses.create(**request)
            validated = CIFExtraction.model_validate_json(response.output_text)
            return {"example_id": example_id, "extracted": validated, "error": None}
        except Exception as exc:
            last_error = exc
            if attempt < max_retries - 1:
                await asyncio.sleep(2 ** attempt)
    return {"example_id": example_id, "extracted": extracted, "error": str(last_error)}


async def validate_cifs_async(
    client: AsyncOpenAI,
    results: list[dict],
    cases_lookup: dict[str, str],
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
    max_concurrency: int = 8,
) -> list[dict]:
    """Validate a batch of extraction results. cases_lookup: example_id → transcript."""
    semaphore = asyncio.Semaphore(max_concurrency)

    async def _one(result: dict, pbar: tqdm) -> dict:
        async with semaphore:
            out = await validate_cif(
                client,
                result["example_id"],
                cases_lookup[result["example_id"]],
                result["extracted"],
                model=model,
                temperature=temperature,
            )
            pbar.update(1)
            if out["error"]:
                pbar.write(f"  WARN {result['example_id']}: {str(out['error'])[:80]}")
            return out

    with tqdm(total=len(results), desc="Validating", unit="case") as pbar:
        return await asyncio.gather(*[_one(r, pbar) for r in results])

## 4. Scoring (V1)

In [10]:
gt_lookup = {c["example_id"]: c["gt"] for c in cases}

In [11]:
_CACHE = Path(f"data/cache/v1_scores_{_MODEL_SLUG}.json")
v1_scores = cache_load_scores(_CACHE)
if v1_scores is None:
    v1_scores = await score_extractions_async(
        client, make_scoring_input(v1_results, gt_lookup),
        model=SCORING_MODEL, temperature=SCORING_TEMPERATURE, max_concurrency=8,
    )
    cache_save_scores(_CACHE, v1_scores)

In [12]:
_cases_lookup = {c["example_id"]: c["transcript"] for c in cases}
_VAL_SLUG = f"{_MODEL_SLUG}_val-{VALIDATION_MODEL.replace('/', '-')}"

_CACHE = Path(f"data/cache/v1_validated_{_VAL_SLUG}.json")
v1_validated = cache_load_extractions(_CACHE)
if v1_validated is None:
    v1_validated = await validate_cifs_async(
        client, v1_results, _cases_lookup,
        model=VALIDATION_MODEL, temperature=VALIDATION_TEMPERATURE,
    )
    cache_save_extractions(_CACHE, v1_validated)

errors = [r for r in v1_validated if r["error"]]
print(f"V1+val: {len(v1_validated) - len(errors)}/{len(v1_validated)} ok")
for r in errors:
    print(f"  {r['example_id']}: {r['error']}")

Validating:   0%|          | 0/68 [00:00<?, ?case/s]

V1+val: 68/68 ok


In [13]:
_CACHE = Path(f"data/cache/v1_validated_scores_{_VAL_SLUG}.json")
v1_validated_scores = cache_load_scores(_CACHE)
if v1_validated_scores is None:
    v1_validated_scores = await score_extractions_async(
        client, make_scoring_input(v1_validated, gt_lookup),
        model=SCORING_MODEL, temperature=SCORING_TEMPERATURE, max_concurrency=8,
    )
    cache_save_scores(_CACHE, v1_validated_scores)

  [1/68] easy_005_inheritance_windfall — 88%
  [2/68] easy_008_inheritance_windfall — 100%
  [3/68] easy_003_high_debt_low_savings — 100%
  [4/68] easy_002_dual_income_mortgage_household — 100%
  [5/68] easy_007_pre_retirement_couple — 95%
  [6/68] easy_001_retired_widowed_client — 95%
  [7/68] easy_004_dual_income_mortgage_household — 92%
  [8/68] easy_006_pre_retirement_couple — 74%
  [9/68] easy_010_inheritance_windfall — 88%
  [10/68] easy_009_dual_income_mortgage_household — 91%
  [11/68] easy_013_self_employed_single — 95%
  [12/68] easy_012_pre_retirement_couple — 92%
  [13/68] easy_014_high_debt_low_savings — 90%
  [14/68] easy_015_messy_corrections_privacy — 92%
  [15/68] easy_011_pre_retirement_couple — 96%
  [16/68] easy_016_pre_retirement_couple — 79%
  [17/68] easy_018_inheritance_windfall — 78%
  [18/68] easy_019_retired_widowed_client — 90%
  [19/68] easy_020_self_employed_single — 95%
  [20/68] hard_001_retired_widowed_client — 91%
  [21/68] hard_003_high_debt_low_savin

In [15]:
m = prf_metrics(v1_scores, gt_lookup)
n = len([r for r in v1_scores if not r.error])
print(f"V1 ({n} cases)  accuracy {m['acc']:.1%}  precision {m['prec']:.1%}  recall {m['rec']:.1%}  F1 {m['f1']:.1%}  hallucination {m['hall']:.1%}")
print()
print(f"  TP={m['tp']}  FN={m['fn']}  FP={m['fp']}  TN={m['tn']}")

V1 (68 cases)  accuracy 90.0%  precision 96.1%  recall 92.8%  F1 94.4%  hallucination 44.8%

  TP=1379  FN=107  FP=56  TN=69


In [27]:
import re as _re

section_counts: dict[str, dict[str, int]] = defaultdict(lambda: dict(tp=0, fn=0, fp=0, tn=0))
for sr in v1_scores:
    if sr.error:
        continue
    gt = gt_lookup[sr.example_id]
    for ss in sr.section_scores:
        base = _re.sub(r'\[\d+\]$', '', ss.section)
        gt_has = gt_section_nonempty(gt, ss.section)
        if   ss.score == 1.0 and     gt_has: section_counts[base]['tp'] += 1
        elif ss.score == 0.0 and     gt_has: section_counts[base]['fn'] += 1
        elif ss.score == 0.0 and not gt_has: section_counts[base]['fp'] += 1
        else:                                section_counts[base]['tn'] += 1

def _pct(n, d): return f"{n/d:.0%}" if d else "—"

print(f"{'Section':<42} {'TP':>5} {'FN':>5} {'Recall':>7}   {'FP':>5} {'TN':>5} {'Halluc%':>8}")
print("─" * 90)
for s in sorted(section_counts):
    c = section_counts[s]
    rec_d  = c['tp'] + c['fn']
    hall_d = c['fp'] + c['tn']
    print(f"{s:<42} {c['tp']:>5} {c['fn']:>5} {_pct(c['tp'], rec_d):>7}   {c['fp']:>5} {c['tn']:>5} {_pct(c['fp'], hall_d):>8}")

Section                                       TP    FN  Recall      FP    TN  Halluc%
──────────────────────────────────────────────────────────────────────────────────────────
client1_employment                            55     5     92%       7     1      88%
client1_personal                              66     2     97%       0     0        —
client2_employment                            43     4     91%       5     0     100%
client2_personal                              50     2     96%       0     0        —
estate_planning                               25     9     74%       0    31       0%
expenses                                     255     6     98%       7     1      88%
has_client2                                   68     0    100%       0     0        —
household                                     37    21     64%       1     9      10%
incomes                                      181    11     94%       0     0        —
loans_and_mortgages                           78 

In [28]:
v1_section_scores = section_scores_from(v1_scores)
print(f"{'Section':<42} {'Accuracy':>8}  {'N':>4}")
print("─" * 90)
for section in sorted(v1_section_scores):
    scores = v1_section_scores[section]
    print(f"{section:<42} {statistics.mean(scores):>8.1%}  {len(scores):>4}")

Section                                    Accuracy     N
──────────────────────────────────────────────────────────────────────────────────────────
client1_employment                            82.4%    68
client1_personal                              97.1%    68
client2_employment                            82.7%    52
client2_personal                              96.2%    52
estate_planning                               86.2%    65
expenses                                      95.4%    63
has_client2                                  100.0%    68
household                                     67.6%    68
incomes                                       94.4%    68
loans_and_mortgages                           84.3%    57
objectives                                    85.4%    68
other_assets                                  71.4%    28
pensions_and_retirement_accounts              97.0%    52
risk_profile_and_preferences                  89.2%    65
savings_and_investments                

## 5. V2 — Per-section Extraction

12 parallel focused calls per transcript, one per CIF section. Each call has a section-specific
system prompt with a strong suppression rule. Sections run in parallel within each transcript;
results are merged into a single `CIFExtraction`.

In [19]:
from src.eval.models import (
    PersonalDetails, EmploymentDetails, HouseholdDetails,
    IncomeItem, ExpenseItem, PensionRetirementItem,
    SavingsInvestmentItem, LoanMortgageItem, OtherAssetItem,
    ObjectiveItem, RiskProfilePreferences, EstatePlanning,
)

def _wrap(key, schema_dict) -> str:
    return json.dumps({"type": "object", "properties": {key: schema_dict}}, indent=2)

def _wrap_list(key, item_model) -> str:
    return _wrap(key, {"type": "array", "items": item_model.model_json_schema()})

def _wrap_obj(key, model) -> str:
    return _wrap(key, model.model_json_schema())

_V2_PROMPT_ROOT = Path("src/prompts/extraction/v2")

SECTIONS_V2 = [
    {
        "key": "personal",
        "output_keys": ["has_client2", "client1_personal", "client2_personal"],
        "schema": json.dumps({
            "type": "object",
            "properties": {
                "has_client2": {"type": ["boolean", "null"], "description": "true if a second client is present"},
                "client1_personal": PersonalDetails.model_json_schema(),
                "client2_personal": {**PersonalDetails.model_json_schema(),
                                     "description": "populate only if has_client2=true, else null"},
            },
        }, indent=2),
    },
    {
        "key": "employment",
        "output_keys": ["client1_employment", "client2_employment"],
        "schema": json.dumps({
            "type": "object",
            "properties": {
                "client1_employment": EmploymentDetails.model_json_schema(),
                "client2_employment": {**EmploymentDetails.model_json_schema(),
                                       "description": "populate only if a second client is present"},
            },
        }, indent=2),
    },
    {"key": "household",    "output_keys": ["household"],                        "schema": _wrap_obj("household",                    HouseholdDetails)},
    {"key": "incomes",      "output_keys": ["incomes"],                          "schema": _wrap_list("incomes",                      IncomeItem)},
    {"key": "expenses",     "output_keys": ["expenses"],                         "schema": _wrap_list("expenses",                     ExpenseItem)},
    {"key": "pensions",     "output_keys": ["pensions_and_retirement_accounts"], "schema": _wrap_list("pensions_and_retirement_accounts", PensionRetirementItem)},
    {"key": "savings",      "output_keys": ["savings_and_investments"],          "schema": _wrap_list("savings_and_investments",      SavingsInvestmentItem)},
    {"key": "loans",        "output_keys": ["loans_and_mortgages"],              "schema": _wrap_list("loans_and_mortgages",          LoanMortgageItem)},
    {"key": "other_assets", "output_keys": ["other_assets"],                     "schema": _wrap_list("other_assets",                 OtherAssetItem)},
    {"key": "objectives",   "output_keys": ["objectives"],                       "schema": _wrap_list("objectives",                   ObjectiveItem)},
    {"key": "risk_profile", "output_keys": ["risk_profile_and_preferences"],     "schema": _wrap_obj("risk_profile_and_preferences", RiskProfilePreferences)},
    {"key": "estate_planning", "output_keys": ["estate_planning"],               "schema": _wrap_obj("estate_planning",              EstatePlanning)},
]

_v2_user_template = load_prompt_yaml(_V2_PROMPT_ROOT / "user.yaml")["prompt"]

def _load_section_system(section: dict) -> str:
    raw = load_prompt_yaml(_V2_PROMPT_ROOT / section["key"] / "system.yaml")["prompt"]
    return raw.format(schema=section["schema"])

SECTION_SYSTEMS_V2 = {s["key"]: _load_section_system(s) for s in SECTIONS_V2}
print(f"{len(SECTION_SYSTEMS_V2)} section prompts loaded")

12 section prompts loaded


In [20]:
async def _extract_section(
    client: AsyncOpenAI,
    section: dict,
    transcript: str,
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
    max_retries: int = 3,
) -> dict:
    """Extract one CIF section. Returns a partial dict with the section's output keys."""
    request = {
        "model": model,
        "input": [
            {"role": "system", "content": SECTION_SYSTEMS_V2[section["key"]]},
            {"role": "user",   "content": _v2_user_template.format(key=section["key"], transcript=transcript)},
        ],
        "temperature": temperature,
        "text": {"format": {"type": "json_object"}},
        "timeout": 60.0,
    }
    last_error = None
    for attempt in range(max_retries):
        try:
            response = await client.responses.create(**request)
            return json.loads(response.output_text)
        except Exception as exc:
            last_error = exc
            if attempt < max_retries - 1:
                await asyncio.sleep(2 ** attempt)
    raise RuntimeError(f"Section '{section['key']}' failed: {last_error!r}")


def _merge_sections(partial_results: list[dict | Exception]) -> CIFExtraction:
    """Combine per-section dicts into a single CIFExtraction."""
    merged: dict = {
        "client1": {"personal": {}, "employment": {}},
        "client2": {"personal": {}, "employment": {}},
    }
    for result in partial_results:
        if isinstance(result, Exception):
            continue
        for key, val in result.items():
            if val is None:
                continue  # let Pydantic default_factory handle missing object sections
            if key == "client1_personal":    merged["client1"]["personal"]    = val or {}
            elif key == "client2_personal":  merged["client2"]["personal"]    = val or {}
            elif key == "client1_employment":merged["client1"]["employment"]  = val or {}
            elif key == "client2_employment":merged["client2"]["employment"]  = val or {}
            else:                            merged[key] = val
    return CIFExtraction.model_validate(merged)


async def extract_cif_v2(
    client: AsyncOpenAI,
    example_id: str,
    transcript: str,
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
) -> dict:
    """Run all section extractions in parallel and merge."""
    cleaned = strip_front_matter(transcript)
    tasks = [_extract_section(client, s, cleaned, model=model, temperature=temperature) for s in SECTIONS_V2]
    results = await asyncio.gather(*tasks, return_exceptions=True)
    errors = [r for r in results if isinstance(r, Exception)]
    try:
        extracted = _merge_sections(results)
        return {
            "example_id": example_id,
            "extracted": extracted,
            "error": str(errors) if errors else None,
        }
    except Exception as exc:
        return {"example_id": example_id, "extracted": CIFExtraction(), "error": str(exc)}


async def extract_cifs_v2_async(
    client: AsyncOpenAI,
    cases: list[dict],
    *,
    model: str = "gpt-5.4-mini",
    temperature: float = 0.0,
    max_concurrency: int = 4,
) -> list[dict]:
    """Batch V2 extraction. max_concurrency limits simultaneous transcripts;
    sections within each transcript run in parallel."""
    semaphore = asyncio.Semaphore(max_concurrency)

    async def _one(case: dict, pbar: tqdm) -> dict:
        async with semaphore:
            result = await extract_cif_v2(
                client, case["example_id"], case["transcript"], model=model, temperature=temperature
            )
            pbar.update(1)
            if result["error"]:
                pbar.write(f"  WARN {case['example_id']}: {str(result['error'])[:80]}")
            return result

    with tqdm(total=len(cases), desc="V2 Extracting", unit="case") as pbar:
        return await asyncio.gather(*[_one(c, pbar) for c in cases])

In [21]:
_CACHE = Path(f"data/cache/v2_results_{_MODEL_SLUG}.json")
v2_results = cache_load_extractions(_CACHE)
if v2_results is None:
    v2_results = await extract_cifs_v2_async(
        client, cases, model=EXTRACTION_MODEL, temperature=EXTRACTION_TEMPERATURE
    )
    cache_save_extractions(_CACHE, v2_results)

errors = [r for r in v2_results if r["error"]]
print(f"V2: {len(v2_results) - len(errors)}/{len(v2_results)} ok")
for r in errors:
    print(f"  {r['example_id']}: {r['error']}")

V2: 68/68 ok


In [22]:
_CACHE = Path(f"data/cache/v2_scores_{_MODEL_SLUG}.json")
v2_scores = cache_load_scores(_CACHE)
if v2_scores is None:
    v2_scores = await score_extractions_async(
        client, make_scoring_input(v2_results, gt_lookup),
        model=SCORING_MODEL, temperature=SCORING_TEMPERATURE, max_concurrency=8,
    )
    cache_save_scores(_CACHE, v2_scores)

In [23]:
_CACHE = Path(f"data/cache/v2_validated_{_VAL_SLUG}.json")
v2_validated = cache_load_extractions(_CACHE)
if v2_validated is None:
    v2_validated = await validate_cifs_async(
        client, v2_results, _cases_lookup,
        model=VALIDATION_MODEL, temperature=VALIDATION_TEMPERATURE,
    )
    cache_save_extractions(_CACHE, v2_validated)

errors = [r for r in v2_validated if r["error"]]
print(f"V2+val: {len(v2_validated) - len(errors)}/{len(v2_validated)} ok")
for r in errors:
    print(f"  {r['example_id']}: {r['error']}")

Validating:   0%|          | 0/68 [00:00<?, ?case/s]

V2+val: 68/68 ok


In [24]:
_CACHE = Path(f"data/cache/v2_validated_scores_{_VAL_SLUG}.json")
v2_validated_scores = cache_load_scores(_CACHE)
if v2_validated_scores is None:
    v2_validated_scores = await score_extractions_async(
        client, make_scoring_input(v2_validated, gt_lookup),
        model=SCORING_MODEL, temperature=SCORING_TEMPERATURE, max_concurrency=8,
    )
    cache_save_scores(_CACHE, v2_validated_scores)

  [1/68] easy_001_retired_widowed_client — 100%
  [2/68] easy_003_high_debt_low_savings — 100%
  [3/68] easy_005_inheritance_windfall — 94%
  [4/68] easy_008_inheritance_windfall — 95%
  [5/68] easy_002_dual_income_mortgage_household — 100%
  [6/68] easy_006_pre_retirement_couple — 100%
  [7/68] easy_007_pre_retirement_couple — 96%
  [8/68] easy_009_dual_income_mortgage_household — 100%
  [9/68] easy_010_inheritance_windfall — 93%
  [10/68] easy_011_pre_retirement_couple — 100%
  [11/68] easy_014_high_debt_low_savings — 89%
  [12/68] easy_004_dual_income_mortgage_household — 100%
  [13/68] easy_012_pre_retirement_couple — 100%
  [14/68] easy_015_messy_corrections_privacy — 91%
  [15/68] easy_018_inheritance_windfall — 100%
  [16/68] easy_013_self_employed_single — 83%
  [17/68] easy_019_retired_widowed_client — 89%
  [18/68] hard_001_retired_widowed_client — 87%
  [19/68] easy_016_pre_retirement_couple — 91%
  [20/68] hard_003_high_debt_low_savings — 90%
  [21/68] easy_020_self_employe

In [26]:
variants = [
    ("V1",     v1_scores),
    ("V1+val", v1_validated_scores),
    ("V2",     v2_scores),
    ("V2+val", v2_validated_scores),
]

stats        = {name: prf_metrics(scores, gt_lookup)    for name, scores in variants}
section_data = {name: section_scores_from(scores)       for name, scores in variants}

w = 10
print(f"{'Metric':<25}" + "".join(f"{n:>{w}}" for n, _ in variants))
print("─" * (25 + w * len(variants)))
for metric, key in [("Accuracy", "acc"), ("Precision", "prec"),
                    ("Recall", "rec"), ("F1", "f1"), ("Hallucination rate", "hall")]:
    row = f"{metric:<25}" + "".join(f"{stats[name][key]:>{w}.1%}" for name, _ in variants)
    print(row)

print()
all_sections = sorted(set().union(*[d.keys() for d in section_data.values()]))
print(f"{'Section':<42}" + "".join(f"{n:>{w}}" for n, _ in variants))
print("─" * (42 + w * len(variants)))
for section in all_sections:
    row = f"{section:<42}" + "".join(
        f"{statistics.mean(section_data[name].get(section, [0.0])):>{w}.1%}"
        for name, _ in variants
    )
    print(row)

Metric                           V1    V1+val        V2    V2+val
─────────────────────────────────────────────────────────────────
Accuracy                      90.0%     91.3%     89.5%     91.6%
Precision                     96.1%     97.2%     94.8%     96.3%
Recall                        92.8%     93.4%     93.5%     94.6%
F1                            94.4%     95.3%     94.1%     95.5%
Hallucination rate            44.8%     36.4%     49.4%     44.2%

Section                                           V1    V1+val        V2    V2+val
──────────────────────────────────────────────────────────────────────────────────
client1_employment                             82.4%     86.8%     83.8%     86.8%
client1_personal                               97.1%     97.1%     92.6%     94.1%
client2_employment                             82.7%     88.5%     77.4%     82.7%
client2_personal                               96.2%     96.2%     94.3%     94.2%
estate_planning                        

### Results review

**Best overall variant: V2+val** - highest accuracy (91.6%), recall (94.6%), and F1 (95.5%) across all four variants. The validation stage adds a consistent ~1-2 pp accuracy lift and reduces hallucinations by 5–8 pp regardless of which extraction approach feeds it.

**V1+val vs V2+val trade-off:** V1+val achieves the lowest hallucination rate (36.4%) and highest precision (97.2%), making it the safer choice when false positives are costly. V2+val wins on recall and F1, making it preferable when completeness matters more.

**Where V2 excels over V1:**
- `household` +20 pp (67.6% -> 88.2%) - focused prompting eliminates confusion with other sections
- `incomes` +4 pp (94.4% -> 98.5%) — dedicated income prompt, no schema noise from 14 other sections
- `loans_and_mortgages` +6 pp (87.1% -> 92.7% with val) — explicit rules about monthly payments vs loan records
- `risk_profile_and_preferences` +4 pp after val (88.7% -> 93.1%)

**Where V1 holds or wins:**
- `other_assets` is V2's clearest failure: 43.8% vs V1's 71.4%, validation barely helps (48.1%). The section-isolated prompt loses context about what was already captured in pensions/savings/incomes, leading to misrouted items.
- `client2_employment` V1+val (88.5%) outperforms V2+val (82.7%) - full-transcript context helps identify the second client's role.
- `expenses` and `pensions` slightly favour V1+val.

**Primary remaining risk:** hallucination rates remain high in absolute terms (36–49%). This is largely driven by `client2_employment` (100% hallucination rate — false positives on single-client cases), `loans_and_mortgages` (~93%), and `objectives` (~92%). These sections warrant tighter prompting or a higher-confidence threshold before accepting list items.


## 5. Real-world Holdout Cases

Two additional transcripts provided with the task — no GT labels exist, so we run extraction
and inspect the output directly.

| File | Format | Est. tokens |
|---|---|---|
| `HT Neviswealth Transcript.txt` | `CLIENT1: [HH:MM:SS]` | ~24 000 |
| `Synthetic Transcript from HT Neviswealth.txt` | multi-line turns | ~9 000 |

> The real transcript (~24k tokens) fits within the model context window but is significantly
> longer than our synthetic cases (~2k tokens). V2 will address this with per-section extraction.

In [13]:
HOLDOUT_FILES = {
    "ht_real":      Path("HT Neviswealth Transcript.txt"),
    "ht_synthetic": Path("Synthetic Transcript from HT Neviswealth.txt"),
}

holdout_cases = [
    {
        "example_id": name,
        "transcript": path.read_text(),
    }
    for name, path in HOLDOUT_FILES.items()
]

for c in holdout_cases:
    raw = c["transcript"]
    cleaned = strip_front_matter(raw)
    est_tokens = len(cleaned) // 4
    print(f"{c['example_id']:20}  {len(raw):>7} chars  ~{est_tokens:>6} tokens after stripping")

ht_real                 94719 chars  ~ 23636 tokens after stripping
ht_synthetic            36593 chars  ~  9115 tokens after stripping


In [14]:
print(f"Extracting {len(holdout_cases)} holdout transcripts...\n")
holdout_results = await extract_cifs_async(client, holdout_cases)

holdout_errors = [r for r in holdout_results if r["error"]]
if holdout_errors:
    for e in holdout_errors:
        print(f"ERROR {e['example_id']}: {e['error']}")

Extracting 2 holdout transcripts...



Extracting:   0%|          | 0/2 [00:00<?, ?case/s]

In [15]:
def display_cif(result: dict) -> None:
    cif: CIFExtraction = result["extracted"]
    d = cif.model_dump(mode="json", exclude_none=True)
    print(f"{'='*60}")
    print(f"  {result['example_id']}")
    print(f"{'='*60}")
    print(json.dumps(d, indent=2))
    print()


for result in holdout_results:
    display_cif(result)

  ht_real
{
  "has_client2": true,
  "client1": {
    "personal": {
      "first_name": "Jerome",
      "marital_or_relationship_status": "married"
    },
    "employment": {
      "status": "employed",
      "occupation": "logistics manager",
      "desired_retirement_date": "2025"
    }
  },
  "client2": {
    "personal": {
      "first_name": "Jennifer",
      "marital_or_relationship_status": "married"
    },
    "employment": {
      "status": "part_time",
      "occupation": "customer service associate",
      "employer": "Kroger",
      "desired_retirement_date": "2025"
    }
  },
  "household": {
    "partner_or_spouse_name": "Jennifer",
    "partner_or_spouse_is_client2": true,
    "partner_or_spouse_relationship": "spouse",
    "children_or_dependants": []
  },
  "incomes": [
    {
      "owner": "client1",
      "source_name": "salary",
      "amount": {
        "normalized_amount": 43000.0,
        "currency": "USD",
        "raw_text": "43"
      },
      "frequency": "ann

In [16]:
metrics = [holdout_metrics(r) for r in holdout_results]

ids = [m["example_id"] for m in metrics]
col_w = max(len(i) for i in ids) + 2

print(f"\n{'Metric':<40}" + "".join(f"{i:>{col_w}}" for i in ids))
print("─" * (40 + col_w * len(metrics)))
print(f"{'Completeness':<40}" + "".join(f"{m['completeness']:>{col_w}.1%}" for m in metrics))
print(f"{'Fields populated / total':<40}" + "".join(
    f"{m['fields_populated']}/{m['total_fields']:>{col_w - len(str(m['fields_populated'])) - 1}}"
    for m in metrics))

print()
print(f"{'Scalar sections':<40}" + "".join(" " * col_w for _ in metrics))
for key in sorted({k for m in metrics for k in m["scalar_sections"]}):
    row = f"  {key:<38}"
    for m in metrics:
        val = m["scalar_sections"].get(key)
        row += f"{'✓' if val else ('–' if val is None else '✗'):>{col_w}}"
    print(row)

print()
print(f"{'List section counts':<40}" + "".join(" " * col_w for _ in metrics))
for section in ["incomes", "expenses", "pensions_and_retirement_accounts",
                "savings_and_investments", "loans_and_mortgages", "other_assets", "objectives"]:
    row = f"  {section:<38}" + "".join(f"{m['list_sections'][section]:>{col_w}}" for m in metrics)
    print(row)


Metric                                         ht_real  ht_synthetic
--------------------------------------------------------------------
Field completeness                               69.1%         70.9%
Fields populated / total                154/       223161/       227

Scalar sections populated                                           
  client1.employment                                 ✓             ✓
  client1.personal                                   ✓             ✓
  client2.employment                                 ✓             ✗
  client2.personal                                   ✓             ✗
  estate_planning                                    ✗             ✓
  has_client2                                        ✓             ✓
  household                                          ✓             ✓
  risk_profile_and_preferences                       ✓             ✓

List section item counts                                            
  incomes                      